In [ ]:
## ================================
## 0) Libraries & read metadata
## ================================
library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(stringr)
library(janitor)
library(viridis)   # colour-blind friendly palettes
library(GGally)
library(tidyverse)

In [ ]:
#metadata from Kevin big excel ath_merge
library(googlesheets4)
gs4_deauth()  # Use public, read-only access

In [ ]:
library(googlesheets4)
library(readr)
library(dplyr)
library(stringr)
library(janitor)

# ---- METADATA FROM GOOGLE SHEETS ----
 meta <- read_sheet(
    "https://docs.google.com/REDACTED",
  sheet = "everything",
  col_types = "c"   # read as character, we'll coerce later
) %>%
  mutate(across(everything(), ~ ifelse(. == "NA", NA, .))) %>%  # keep real NAs
  mutate(sample = str_squish(sample),
         tube_id = str_squish(tube_id))

In [ ]:
dim(meta)
glimpse(meta)

In [ ]:
# ---- EXPRESSION MATRIX ----
expr_raw <- read_csv("20250710_RNana_Seqtk_Filtered_vst_limma_w.DesignMat.csv", show_col_types = FALSE)

In [ ]:
dim(expr_raw)
head(expr_raw)

In [ ]:
clean_names <- function(x) {
  x %>%
    str_replace("_[Rr]edo.*$", "") %>%    # remove _redo, _redo1, _Redo2 etc.
    str_replace("_[Vv][0-9]+$", "") %>%   # remove _v1, _V2...
    str_replace("_[Rr][0-9]+$", "") %>%   # remove _r1, _R2...
    str_replace("_[0-9]+$", "")           # remove pure numeric suffixes like _1
}

In [ ]:
expr_clean <- expr_raw
colnames(expr_clean) <- colnames(expr_clean) %>% clean_names()


In [ ]:
colnames(expr_clean)[1:20]

In [ ]:
expr_ids <- colnames(expr_clean)[-1]

In [ ]:
length(expr_ids)   # should be 616
head(expr_ids)

In [ ]:
meta_clean <- meta %>%
  mutate(sample_clean = clean_names(sample))

In [ ]:
head(meta_clean$sample_clean)

In [ ]:
meta_616 <- meta_clean %>%
  filter(sample_clean %in% expr_ids) %>%
  distinct(sample_clean, .keep_all = TRUE)

In [ ]:
nrow(meta_616)
length(unique(meta_616$sample_clean))

In [ ]:
meta_final <- tibble(sample_clean = expr_ids) %>%
  left_join(meta_616, by = "sample_clean")

In [ ]:
dim(meta_final)                     # should be 616 × (metadata columns)
identical(meta_final$sample_clean, expr_ids)   # TRUE

In [ ]:
glimpse(meta_final)

In [ ]:
# basic checks
nrow(meta_final)                        # expect 616
length(unique(meta_final$sample))       # expect 616

In [ ]:
# define df and season_short
meta_final <- meta_final %>%
  mutate(
    season_short = str_replace(season, "^(\\d{4})-\\d{2}-(.*)$", "\\1-\\2")
  )

table(meta_final$season_short)

In [ ]:
library(dplyr)
library(stringr)
library(lubridate)

guess_col_type <- function(x) {
  # Treat empty strings and literal "NA" as missing
  x <- ifelse(x %in% c("", "NA", "NaN", "na", "n/a"), NA, x)
  
  # If everything is NA → leave as character
  if (all(is.na(x))) return(x)
  
  n_non_na <- sum(!is.na(x))
  
  ## ---- Try numeric ----
  num_try <- suppressWarnings(as.numeric(str_replace(x, ",", ".")))
  if (sum(!is.na(num_try)) >= 0.9 * n_non_na) {
    return(num_try)
  }
  
  ## ---- Try logical (TRUE/FALSE, 0/1, yes/no) ----
  logi_map <- function(v) {
    case_when(
      v %in% c("TRUE","True","T","1","yes","Yes","Y")   ~ TRUE,
      v %in% c("FALSE","False","F","0","no","No","N")   ~ FALSE,
      TRUE                                              ~ NA
    )
  }
  logi_try <- logi_map(x)
  if (sum(!is.na(logi_try)) >= 0.9 * n_non_na) {
    return(logi_try)
  }
  
  ## ---- Try Date (ymd or dmy) ----
  d1 <- suppressWarnings(ymd(x))
  d2 <- suppressWarnings(dmy(x))
  date_try <- ifelse(!is.na(d1), d1, d2)
  if (sum(!is.na(date_try)) >= 0.9 * n_non_na) {
    return(as.Date(date_try))
  }
  
  ## ---- Try POSIXct (datetime) ----
  dt1 <- suppressWarnings(ymd_hms(x, tz = "UTC"))
  dt2 <- suppressWarnings(dmy_hms(x, tz = "UTC"))
  dt_try <- ifelse(!is.na(dt1), dt1, dt2)
  if (sum(!is.na(dt_try)) >= 0.9 * n_non_na) {
    return(as.POSIXct(dt_try, origin = "1970-01-01", tz = "UTC"))
  }
  
  ## ---- Fallback: keep as character ----
  return(x)
}


In [ ]:
meta_final_typed <- meta_final %>%
  mutate(across(everything(), guess_col_type))

glimpse(meta_final_typed)


In [ ]:
# Abiotic (weather)
abiotic_vars <- c("mean_temp_4d", "mean_hum_4d",
                  "mean_fluctuation_4d", "precip_30d_sum", "hours_after_sunrise")

# Plant morphology
morpho_vars <- c("max_diam_mm", "n_leaves")

# Presence absence

p_a_vars <- c("bolting","hr")

# Composite field scores
composite_vars <- c("composite_bacterial", "composite_fungi_hpa")

# Microbial load (metagenomic)
microbe_vars <- c("bacterial_per_plant", "fungal_per_plant", "bact_family_entropy")

# Field symptoms
symptom_vars <- c("herbivory")

vars_all <- c(abiotic_vars, morpho_vars,
              composite_vars, microbe_vars,
              symptom_vars, p_a_vars)

# pretty labels for plots
var_labels <- c(
  mean_temp_4d        = "Mean Temperature (°C)",
  mean_hum_4d         = "Mean Relative humidity (%)",
  mean_fluctuation_4d = "Temperature fluctuation (°C)",
  precip_30d_sum      = "Preciptation sum (mm)",
  hours_after_sunrise = "Hours after sunrise",
  max_diam_mm         = "Max diameter (mm)",
  n_leaves            = "Number of leaves",
  composite_bacterial = "Composite bacteria (0-7)",
  composite_fungi_hpa = "Composite fungi/Hpa (0-7)",
  chlorosis           = "Chlorosis (0-5)",
  necrosis            = "Necrosis (0-5)",
  herbivory           = "Herbivory (0-5)",
  fungi               = "Fungi (0-5)",
  albugo              = "Albugo (0-5)",
  hyalo               = "Hyalo (0-5)",
  hr                  = "HR (y/n)",
  bolting             = "Bolting (y/n)",
  bacterial_per_plant = "Bacterial load / plant",
  bact_family_entropy = "Bacterial Family Entropy ",
  fungal_per_plant    = "Fungal load / plant", 
  hyalo_per_plant = "hyalo_per_plant", 
  pseudo_per_plant = "Pseudo load / plant",
  sphingo_per_plant =  "Sphingo load / plant",
  xantho_per_plant = "Xantho load /plant"
)


In [ ]:
names(meta_final_typed)

In [ ]:
# ================================
## Fig 2A – Trait correlations by season (panel-ready)
## ================================

cor_long_season <- meta_final_typed%>%
  group_by(season_short) %>%
  group_modify(~ {
    mat <- .x %>%
      select(all_of(vars_all)) %>%
      cor(use = "pairwise.complete.obs")

    # keep ONLY lower triangle, remove upper + diagonal
    mat[upper.tri(mat, diag = TRUE)] <- NA

    as.data.frame(mat) %>%
      mutate(var1 = rownames(mat)) %>%
      pivot_longer(cols = -var1,
                   names_to = "var2",
                   values_to = "cor")
  }) %>%
  ungroup() %>%
  filter(!is.na(cor)) %>%
  mutate(
    var1    = factor(var1, levels = vars_all),      # no rev() here for lower tri
    var2    = factor(var2, levels = vars_all),
    abs_cor = abs(cor)
  )


In [ ]:

label_thresh <- 0.4

p_fig2A <- ggplot(cor_long_season,
                  aes(x = var2, y = var1)) +
  geom_point(aes(size = abs_cor, fill = cor),
             shape = 21, colour = "grey70") +
  geom_text(
    data = subset(cor_long_season, abs_cor >= label_thresh),
    aes(label = sprintf("%.2f", cor)),
    size = 1.7,
    colour = "black"
  ) +
  scale_size_continuous(range = c(0, 5.5), guide = "none") +
  scale_fill_gradient2(
    low    = "#2166ac",
    mid    = "white",
    high   = "#b2182b",
    limits = c(-1, 1),
    name   = "Correlation"
  ) +
  facet_wrap(~ season_short, nrow = 1) +
  scale_x_discrete(labels = var_labels) +
  scale_y_discrete(labels = var_labels) +
  coord_fixed() +
  theme_bw(base_size = 10) +
  theme(
    strip.text      = element_text(size = 6, face = "bold"),
    axis.text.x     = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
    axis.text.y     = element_text(size = 7),
    axis.title      = element_blank(),
    panel.grid      = element_blank(),
    panel.border    = element_rect(colour = "black", fill = NA, linewidth = 0.3),
    legend.position = "right"
  ) +
  ggtitle("")

p_fig2A


In [ ]:
## Save as PNG (for drafts / slides)
ggsave(
  filename = "Fig2A_trait_correlations.png",
  plot     = p_fig2A,
  width    = 10,
  height   = 5,
  dpi      = 300
)

In [ ]:
## ================================
## Fig 2B – Stacked barplots of composite scores
## ================================

library(dplyr)
library(ggplot2)
library(tidyr)
library(stringr)
library(viridis)


# prepare data: one row = population × season × country × score category
scores_long <- meta_final_typed%>%
  filter(!is.na(composite_bacterial),
         country %in% c("DE", "FR", "US")) %>%
  mutate(
    # treat composite score as a categorical factor (0–9)
    score_cat = factor(composite_bacterial,
                       levels = sort(unique(composite_bacterial)))
  ) %>%
  group_by(season_short, country, population, score_cat) %>%
  summarise(
    n = n(),
    .groups = "drop_last"
  ) %>%
  mutate(
    prop = n / sum(n)   # proportion within each population
  ) %>%
  ungroup()

# sanity check
scores_long %>% count(season_short, country)   # see how many pops per facet

p_fig2B_stacked <- ggplot(scores_long,
                          aes(x = population, y = prop, fill = score_cat)) +
  geom_col(width = 0.8, colour = "grey10", linewidth = 0.2) +
  scale_y_continuous(
    labels = scales::percent_format(accuracy = 1),
    expand = expansion(mult = c(0, 0.02))
  ) +
  scale_fill_viridis_d(
    option = "viridis",
    direction = 1,
    name = "Composite\nscore"
  ) +
  facet_grid(
    season_short ~ country,
    scales = "free_x",
    space  = "free_x"
  ) +
  theme_bw(base_size = 10) +
  theme(
    axis.text.x       = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
    legend.position   = "right",
    strip.text        = element_text(size = 11, face = "bold"),
    panel.grid.major.x = element_blank(),
    panel.grid.minor  = element_blank()
  ) +
  labs(
    title = "",
    x     = "Population",
    y     = "Proportion of plants"
  )

p_fig2B_stacked


In [ ]:
## Save as PNG (for drafts / slides)
ggsave(
  filename = "Fig2B_composite_bacterial.png",
  plot     = p_fig2B_stacked,
  width    = 10,
  height   = 5,
  dpi      = 300
)

In [ ]:
## ================================
## Fig 2C – Abiotic environment (Season × Country, no hours_after_sunrise)
## ================================

library(rlang)
library(patchwork)

abiotic_vars_c <- c("mean_temp_4d", "mean_hum_4d",
                    "mean_fluctuation_4d", "precip_30d_sum")


make_abiotic_plot <- function(var_name) {
  v_sym <- sym(var_name)
  
  ggplot(
    meta_final_typed %>% filter(country %in% c("DE", "FR", "US")),
    aes(x = season_short, y = !!v_sym, fill = country)
  ) +
    geom_violin(
      position = position_dodge(width = 0.8),
      trim = FALSE,
      alpha = 0.5,
      colour = "grey60"
    ) +
    geom_boxplot(
      position = position_dodge(width = 0.8),
      width = 0.18,
      alpha = 0.9,
      outlier.size = 0.7
    ) +
    scale_fill_viridis_d(option = "magma", end = 0.85, name = "Country") +
    theme_bw(base_size = 10) +
    theme(
      legend.position = "right",
      axis.text.x     = element_text(angle = 90, hjust = 1, vjust = 0.5),
      panel.grid      = element_blank()
    ) +
    labs(
      #title = var_labels[[var_name]],
      x     = "",
      y     = var_labels[[var_name]]
    )
}

p_temp   <- make_abiotic_plot("mean_temp_4d")
p_hum    <- make_abiotic_plot("mean_hum_4d")
p_fluct  <- make_abiotic_plot("mean_fluctuation_4d")
p_precip <- make_abiotic_plot("precip_30d_sum")

## combined Fig. 1C (2x2 layout)
p_fig2C_combined <- (p_temp | p_hum) /
                    (p_fluct | p_precip) +
  plot_annotation(
    title = ""
  )

p_fig2C_combined



In [ ]:
## Save as PNG (for drafts / slides)
ggsave(
  filename = "Fig2C_Weather_summaries.png",
  plot     = p_fig2C_combined,
  width    = 5,
  height   = 5,
  dpi      = 300
)

In [ ]:
## ================================
## Fig 2D – Plant size (diameter, leaves)
## ================================


make_morpho_plot <- function(var_name) {
  v_sym <- sym(var_name)
  
  ggplot(
    meta_final_typed %>% filter(country %in% c("DE", "FR", "US")),
    aes(x = season_short, y = !!v_sym, fill = country)
  ) +
    geom_violin(
      position = position_dodge(width = 0.8),
      trim = FALSE,
      alpha = 0.5,
      colour = "grey60"
    ) +
    geom_boxplot(
      position = position_dodge(width = 0.8),
      width = 0.18,
      alpha = 0.9,
      outlier.size = 0.8
    ) +
    scale_fill_viridis_d(option = "magma", end = 0.8, name = "Country") +
    theme_bw(base_size = 10) +
    theme(
      legend.position = "right",
      panel.grid      = element_blank(),
      axis.text.x     = element_text(angle = 45, hjust = 1)
    ) +
    labs(
      title = var_labels[[var_name]],
      x     = "Season",
      y     = var_labels[[var_name]]
    )
}

p_diameter <- make_morpho_plot("max_diam_mm")
p_leaves   <- make_morpho_plot("n_leaves")

# Combine 2 × 1 layout using patchwork
p_fig2D_combined <- (p_diameter | p_leaves) +
  plot_annotation(
    title = ""
  )

p_fig2D_combined



In [ ]:
## Save as PNG (for drafts / slides)
ggsave(
  filename = "Fig2D_plant_size.png",
  plot     = p_fig2D_combined,
  width    = 5,
  height   = 5,
  dpi      = 300
)

In [ ]:
## ================================
## Fig 2E – Field symptom scores (y from 0 to 5)
## ================================

library(rlang)
library(patchwork)
library(ggplot2)

make_symptom_plot <- function(var_name) {
  v_sym <- sym(var_name)
  
  ggplot(
    meta_final_typed %>% filter(country %in% c("DE", "FR", "US")),
    aes(x = season_short, y = !!v_sym, fill = country)
  ) +
    geom_violin(
      position = position_dodge(width = 0.8),
      trim = FALSE,
      alpha = 0.5,
      colour = "grey60"
    ) +
    geom_boxplot(
      position = position_dodge(width = 0.8),
      width = 0.18,
      alpha = 0.9,
      outlier.size = 0.7
    ) +
    coord_cartesian(ylim = c(0, 5)) +
    scale_fill_viridis_d(option = "magma", end = 0.85, name = "Country") +
    theme_bw(base_size = 10) +
    theme(
      axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5),
      panel.grid  = element_blank()
      # axis titles will be set per-panel later
    ) +
    labs(
      title = var_labels[[var_name]],
      x     = NULL,
      y     = NULL
    )
}

p_chl   <- make_symptom_plot("chlorosis") + labs(y = "Score", x = NULL)
p_nec   <- make_symptom_plot("necrosis")  + labs(y = NULL,    x = NULL)
p_herb  <- make_symptom_plot("herbivory") + labs(y = NULL,    x = NULL)

p_fungi <- make_symptom_plot("fungi")     + labs(y = "Score", x = NULL)
p_alb   <- make_symptom_plot("albugo")    + labs(y = NULL,    x = NULL)
p_hyal  <- make_symptom_plot("hyalo")     + labs(y = NULL,    x = NULL)

p_fig2E_combined <- (p_chl | p_nec | p_herb) /
                    (p_fungi | p_alb | p_hyal) +
  plot_annotation(title = "") +
  plot_layout(guides = "collect") & 
  theme(legend.position = "bottom")

p_fig2E_combined



In [ ]:
## Save as PNG (for drafts / slides)
ggsave(
  filename = "Fig2E_Field_scores.png",
  plot     = p_fig2E_combined,
  width    = 5,
  height   = 5,
  dpi      = 300
)

In [ ]:
## ================================
## Summary table by season
## ================================

summary_table <- meta_final_typed %>%
  group_by(season_short) %>%
  summarise(
    n_samples          = n(),
    mean_temp_4d       = mean(mean_temp_4d,       na.rm = TRUE),
    mean_hum_4d        = mean(mean_hum_4d,        na.rm = TRUE),
    mean_fluct_4d      = mean(mean_fluctuation_4d,na.rm = TRUE),
    mean_precip_30d    = mean(precip_30d_sum,     na.rm = TRUE),
    mean_hours_sunrise = mean(hours_after_sunrise,na.rm = TRUE),
    mean_max_diam      = mean(max_diam_mm,        na.rm = TRUE),
    mean_n_leaves      = mean(n_leaves,           na.rm = TRUE),
    mean_comp_bact     = mean(composite_bacterial,na.rm = TRUE),
    mean_herbivory     = mean(herbivory,          na.rm = TRUE)
  )

summary_table


In [ ]:
### suplementary

In [ ]:
## ================================
##  Composite bacteria (barplot per population)
## ================================

comp_bact_pop <- meta_final_typed %>%
  group_by(season_short, country, population) %>%
  summarise(
    n    = n(),
    mean = mean(composite_bacterial, na.rm = TRUE),
    sd   = sd(composite_bacterial, na.rm = TRUE),
    se   = sd / sqrt(n),
    .groups = "drop"
  )

# quick check: how many populations per facet?
comp_bact_pop %>%
  count(season_short, country)

p_figSup1 <- ggplot(comp_bact_pop,
                  aes(x = population, y = mean, fill = country)) +
  geom_col(width = 0.7, colour = "black") +
  geom_errorbar(aes(ymin = mean - se, ymax = mean + se),
                width = 0.2, linewidth = 0.3) +
  scale_fill_viridis_d(option = "magma", end = 0.85, name = "Country") +
  facet_grid(season_short ~ country, scales = "free_x", space = "free_x") +
  theme_bw(base_size = 11) +
  theme(
    axis.text.x     = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
    legend.position = "none",
    strip.text      = element_text(size = 10, face = "bold"),
    panel.grid.major.x = element_blank()
  ) +
  labs(
    title = "Mean Composite bacterial score per population",
    x     = "Population",
    y     = "Mean composite bacterial score"
  )

p_figSup1

In [ ]:
## Save as PNG (for drafts / slides)
ggsave(
  filename = "Fig_Supplementary1.png",
  plot     = p_figSup1,
  width    = 10,
  height   = 5,
  dpi      = 300
)